# Page-pair text model — document break + page order (Colab first)

Briefs 1 and 2, steps E3 / Q3. One encoder (**Bio_ClinicalBERT** by default) reads the **end of page A** and the **start of page B** (about 150 words each) and has two heads:

* **break** — P(the document ends after page A) at every meeting point, every document type (`document_pairs.jsonl`, built by `continuity/ml/build_data.py`);
* **follows** — P(page B comes directly after page A), for pages at most 5 places apart in one date-of-service block (`order_pairs.jsonl`).

Its two scores become one more clue for the continuity and page-order clue models. **Colab:** Runtime → Change runtime type → **GPU** (T4 or better), edit `BASE_PATH` only, then Runtime → Run all.

Put these in one Drive folder (`BASE_PATH`):

* `train_pairs.py`, `requirements.txt` — from `training/bert-training-boundary/`
* `document_pairs.jsonl` (built by `core-pipeline/stages/lib/continuity/ml/build_data.py`) and `order_pairs.jsonl` (the ±5 pairs from `Training/boundary/local/order_pairs_local.jsonl`)

`page-pair-training.zip` in `training/bert-training-boundary/` holds all four. Both files are already split **by chart** (train / val / test); test is scored once, never tuned on. Page text is never printed.

In [ ]:
import json, os, sys, subprocess, time
from pathlib import Path

ON_COLAB = 'google.colab' in sys.modules
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/page-pair-training')  # <- edit: the Drive folder
    CODE, DATA = BASE_PATH, BASE_PATH
else:
    CODE = Path.cwd()                                               # run from training/bert-training-boundary
    DATA = CODE / 'data'
OUTPUT = CODE / 'output' / 'boundary-text'

BASE_MODEL = 'emilyalsentzer/Bio_ClinicalBERT'   # or 'distilbert-base-uncased' (faster, a little weaker)
EPOCHS, BATCH_SIZE, LR = 4, 16, 2e-5
FOLDS = 5          # out-of-fold text scores for the clue models (5 more short trainings); 0 to skip

for name in ('train_pairs.py', 'requirements.txt'):
    assert (CODE / name).is_file(), f'{name} is missing from {CODE}'
for name in ('document_pairs.jsonl', 'order_pairs.jsonl'):
    assert (DATA / name).is_file(), f'{name} is missing from {DATA}'
print('code:', CODE, '| data:', DATA, '| output:', OUTPUT, '| base:', BASE_MODEL)

In [ ]:
if ON_COLAB:
    %pip install -q -r {str(CODE / 'requirements.txt')}

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
def run(*args):
    """Run train_pairs.py with these paths; its output streams below."""
    cmd = [sys.executable, str(CODE / 'train_pairs.py'),
           '--document-pairs', str(DATA / 'document_pairs.jsonl'), '--order-pairs', str(DATA / 'order_pairs.jsonl'),
           '--output', str(OUTPUT), '--base-model', BASE_MODEL,
           '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE), '--lr', str(LR), *args]
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env, cwd=CODE) as proc:
        for line in proc.stdout:
            print(line, end='', flush=True)
    if proc.returncode:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

## Check

Label counts per head and split, two training steps and a time estimate. Saves nothing. Out of memory: set `BATCH_SIZE = 8`.

In [ ]:
run('--check', '--folds', str(FOLDS))

## Train, score, export

Trains on the train charts, keeps the best epoch by mean validation AUC of the two heads, scores the test charts once, exports `model.onnx` (CPU), then — with `FOLDS` — trains `FOLDS` more models grouped by chart so every train / val pair gets a text score from a model that never saw it (`scores_break.csv`, `scores_follows.csv`).

In [ ]:
run('--folds', str(FOLDS))

## Results on the test charts

Targets from the briefs: the clue models alone are at F1 0.82 (document break, all types); the goal with this score added is 0.85+. These numbers are the text model **alone**.

In [ ]:
metrics = json.loads((OUTPUT / 'metrics.json').read_text())
print(json.dumps(metrics['test'], indent=1))
print('validation (best epoch):', json.dumps(metrics['val_best']))
print('train / val scores:', metrics['scores'])

## ONNX check

The exported model gives the same probabilities as PyTorch, and the time per meeting point on CPU (the brief's limit: under 150 ms).

In [ ]:
%pip install -q onnxruntime
import numpy as np, onnxruntime as ort
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(OUTPUT / 'tokenizer')
space = json.loads((OUTPUT / 'label_space.json').read_text())
sess = ort.InferenceSession(str(OUTPUT / 'model.onnx'), providers=['CPUExecutionProvider'])
a = ' '.join(['word'] * space['words_per_side']); b = ' '.join(['next'] * space['words_per_side'])
enc = tok([a], [b], truncation='longest_first', max_length=space['max_length'], return_tensors='np')
feed = {'input_ids': enc['input_ids'].astype(np.int64), 'attention_mask': enc['attention_mask'].astype(np.int64)}
sess.run(None, feed)
t = time.perf_counter(); n = 20
for _ in range(n): sess.run(None, feed)
print(f'CPU: {1000 * (time.perf_counter() - t) / n:.0f} ms per meeting point (full-length input); probs', sess.run(None, feed)[0].round(3))

## Inference check

Paste the end of one page and the start of the next.

In [ ]:
TAIL_A = """
paste the last lines of page A here
"""
HEAD_B = """
paste the first lines of page B here
"""
enc = tok([TAIL_A.strip()], [HEAD_B.strip()], truncation='longest_first', max_length=space['max_length'], return_tensors='np')
p_break, p_follows = sess.run(None, {'input_ids': enc['input_ids'].astype(np.int64), 'attention_mask': enc['attention_mask'].astype(np.int64)})[0][0]
print(f'P(document ends after A) {p_break:.3f}   P(B comes right after A) {p_follows:.3f}')

## Take it back

Zips `model.onnx`, `tokenizer/`, `label_space.json`, `metrics.json` and the two score files next to the run on Drive (the PyTorch weights stay in `output/`). Download it and unzip to `core-pipeline/models/boundary-text/` on the laptop; the clue models take the scores as one more clue (E3 / Q3).

In [ ]:
import shutil, zipfile
archive = CODE / 'output' / 'boundary-text.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT.rglob('*'):
        if p.is_file() and not (p.name == 'heads.pt' or 'encoder' in p.relative_to(OUTPUT).parts):
            z.write(p, p.relative_to(OUTPUT))
print('wrote', archive, f'({archive.stat().st_size / 1e6:.0f} MB)')